In [1]:
import pandas as pd
import numpy as np
df = pd.read_parquet('../Data/Risk_Dataset_clean.parquet')
df.head()

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies,emp_length_num
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3,5.0
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0,1.0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0,4.0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0,8.0
5,CUST_00006,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,...,-122.4194,Full-time,36,0.252525,2211.741134,0.475933,10,0.827034,0,2.0



## Thêm các FE cần thiết và ý nghĩa  ##
1 installment	Số tiền phải trả hàng tháng → đo gánh nặng trả nợ
2 installment_income_ratio	Tỷ lệ thu nhập dành cho trả nợ → càng cao càng rủi ro
3 Tổng nghĩa vụ nợ sau khi vay 

1.  Installment Amount: Giá trị khoản vay chia cho thời hạn vay
2. Installment-to-Income Ratio:
3. Total Debt After Loan: đo mức độ “ngộp nợ”
4.  credit_age_bucket	Nhóm tuổi tín dụng → lịch sử tín dụng càng dài càng ít rủi ro
5. delinquency_severity	Mức độ trễ hạn → phân loại rủi ro hành vi
6. has_delinquency	Flag từng trễ hạn → biến nhị phân mạnh
7. utilization_bucket	Nhóm mức sử dụng tín dụng → sử dụng cao = rủi ro cao
8. loan_grade_numeric	Mã hóa loan grade theo thứ tự rủi ro
9. interest_rate_bucket	Nhóm lãi suất → lãi cao thường đi kèm rủi ro cao
10. emp_stability	Độ ổn định nghề nghiệp so với lịch sử tín dụng
11. risk_interaction	Tương tác giữa DTI và utilization → rủi ro tăng theo cấp số nhân

In [2]:
import pandas as pd
import numpy as np


In [3]:
# 1. Installment Amount
df["installment"] = df["loan_amnt"] / df["loan_term_months"]


In [4]:
# 2. Installment-to-Income Ratio
df["installment_income_ratio"] = df["installment"] / df["person_income"]

In [5]:
# 3. Total Debt After Loan
df["total_debt_after_loan"] = df["other_debt"] + df["loan_amnt"]

In [6]:
# 5. Credit Age Bucket
def credit_age_bucket(x):
    if x < 1:
        return "very_short"
    elif x < 3:
        return "short"
    elif x < 5:
        return "medium"
    else:
        return "long"

df["credit_age_bucket"] = df["cb_person_cred_hist_length"].apply(credit_age_bucket)

In [7]:
#6
df['age_bucket'] = pd.cut(
    df['person_age'],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
)

In [8]:
#7
df['income_bucket'] = pd.cut(
    df['person_income'],
    bins=[0, 30000, 60000, 100000, 200000, df['person_income'].max()],
    labels=['<30k', '30-60k', '60-100k', '100-200k', '200k+']
)

In [9]:
# 8. Has Delinquency Flag
df["has_delinquency"] = (df["past_delinquencies"] > 0).astype(int)

In [10]:
# 9. Credit Utilization Bucket
def utilization_bucket(x):
    if x < 0.3:
        return "low"
    elif x < 0.6:
        return "medium"
    elif x < 0.9:
        return "high"
    else:
        return "critical"

df["utilization_bucket"] = df["credit_utilization_ratio"].apply(utilization_bucket)

In [11]:
# 10. Loan Grade Numeric
grade_map = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7}
df["loan_grade_numeric"] = df["loan_grade"].map(grade_map)

In [12]:
#11
df['interest_rate_bucket'] = pd.cut(
    df['loan_int_rate'],
    bins=[0, 5, 10, 15, 20, 30],
    labels=['<5%', '5-10%', '10-15%', '15-20%', '20%']
)

In [13]:
#12
df['loan_amount_bucket'] = pd.cut(
   df['loan_amnt'],
    bins=[0, 2000, 5000, 10000, 20000, df['loan_amnt'].max()],
    labels=['<2k', '2-5k', '5-10k', '10-20k', '20k+'])

In [14]:
# 13. Employment Stability Score
df["emp_stability"] = df["person_emp_length"] / df["cb_person_cred_hist_length"]
df["emp_stability"].replace([np.inf, -np.inf], np.nan, inplace=True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_22152\3704368862.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["emp_stability"].replace([np.inf, -np.inf], np.nan, inplace=True)


1        2.500000
2        0.333333
3        2.000000
4        2.000000
5        1.000000
           ...   
32576    0.033333
32577    0.210526
32578    0.107143
32579    0.192308
32580    0.066667
Name: emp_stability, Length: 31679, dtype: float64

In [15]:
# 12. Risk Interaction (DTI × Utilization)
df["risk_interaction"] = df["debt_to_income_ratio"] * df["credit_utilization_ratio"]

In [16]:
df.shape

(31679, 43)

In [17]:
df.columns

Index(['client_ID', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length', 'gender',
       'marital_status', 'education_level', 'country', 'state', 'city',
       'city_latitude', 'city_longitude', 'employment_type',
       'loan_term_months', 'loan_to_income_ratio', 'other_debt',
       'debt_to_income_ratio', 'open_accounts', 'credit_utilization_ratio',
       'past_delinquencies', 'emp_length_num', 'installment',
       'installment_income_ratio', 'total_debt_after_loan',
       'credit_age_bucket', 'age_bucket', 'income_bucket', 'has_delinquency',
       'utilization_bucket', 'loan_grade_numeric', 'interest_rate_bucket',
       'loan_amount_bucket', 'emp_stability', 'risk_interaction'],
      dtype='str')

In [18]:
df.head()

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,credit_age_bucket,age_bucket,income_bucket,has_delinquency,utilization_bucket,loan_grade_numeric,interest_rate_bucket,loan_amount_bucket,emp_stability,risk_interaction
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,short,18-25,<30k,1,medium,2,10-15%,<2k,2.500000,0.159031
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,medium,18-25,<30k,0,high,3,10-15%,5-10k,0.333333,0.645982
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,short,18-25,60-100k,0,medium,3,15-20%,20k+,2.000000,0.244136
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,medium,18-25,30-60k,0,low,3,10-15%,20k+,2.000000,0.212279
5,CUST_00006,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,...,short,18-25,<30k,0,high,1,5-10%,2-5k,1.000000,0.393613


In [19]:
df.shape

(31679, 43)

In [20]:
df.to_csv('../Data/Risk_data_FE.csv')